# 회고

제일좋은 모델은 .. 이었다


# 실험 설계
- AI 한번 돌리는데 너무 많은 연산과 컴퓨팅 파워가 소모된다...돈없는 사람은 AI 도 못쓰나?
- 단순히 점수 가장 높은 모델이 아니라, **적은 메모리와 연산**으로 어느 정도까지 쓸 만한 성능을 낼 수 있는지를 확인 하자
- 이를 위해 여러 ML 모델과 Dense 딥러닝 모델을, 같은 TF‑IDF 표현 위에서, 단어장(vocabulary) 크기를 바꿔가며 비교


### 1. 실험 목표

1. 단어장 크기(vocabulary size)에 따른 성능·효율 변화 관찰 
   - `max_features = 10000, 5000, None(All)`로 단어장 크기를 바꿨을 때  
   - Accuracy와 F1‑score가 어떻게 달라지는지 확인  
   - 성능을 거의 잃지 않고 줄일 수 있는 최소 단어장 크기에 대한 감을 잡기

2. 같은 TF‑IDF 표현에서 여러 ML 모델의 특성 비교
   - Logistic Regression, SVM(LinearSVC), RandomForest, XGBoost, Naive Bayes, LightGBM, DecisionTree, Dense NN  
   - 각 모델의 Accuracy, F1‑score를 비교하고  
   - 어떤 모델이 가볍고 빠르면서도 성능이 괜찮은지 파악

3. 딥러닝(Dense) vs 전통 머신러닝의 차이 이해
   - 같은 TF‑IDF 입력을 사용했을 때, Dense 모델과 전통 ML(Logistic Regression, SVM 등)의 성능과 효율을 비교  

4. 저성능, 개인용 컴퓨터 환경을 염두에 둔 경량 모델 감각 익히기
   - 제한된 메모리와 연산 자원(로컬 LLM, 온디바이스 AI 등)을 고려했을 때  
   - TF‑IDF + 전통 ML 조합이 얼마나 강력한 베이스라인인지 체감  
   - 성능 vs 모델 크기 vs 속도 사이의 트레이드오프를 확인

---

### 2. 실험 설계

- 데이터셋 : Keras Reuters 뉴스 데이터 (다중 분류, 46 클래스)  
- 입력 표현:  
  - 인덱스를 단어로 디코딩한 후  `CountVectorizer(max_features=...)` + `TfidfTransformer()`로 TF‑IDF 벡터화  
- Vocabulary Size:  
  - 2,000/ 10,000 / 5,000 / None(All words)  
- 비교할 모델 (총 8개):  
  - Logistic Regression  
  - SVM (LinearSVC)  
  - RandomForest  
  - XGBoost  
  - Naive Bayes (MultinomialNB)  
  - Dense NN (Keras)  
  - LightGBM  
  - DecisionTree  
- 평가지표:  
  - Accuracy  
  - F1‑score (다중 분류 → macro 또는 weighted, 실험 시 명시)
  - Train Time (s): `fit()`에 걸린 학습 시간 (초 단위 측정)  
  - Test Time (s):  전체 테스트 데이터에 대한 `predict()` 시간  
  - Model Size (rough):  
    - 전통 ML: 학습된 모델 객체를 디스크에 저장했을 때 용량 (예: `joblib.dump` 후 파일 크기)  
    - Dense NN: `model.count_params()`로 파라미터 수, 또는 저장된 `.h5` 파일 크기  

---

### 3. 기대하는 인사이트

- 단어장 크기를 줄여도 성능이 거의 유지되는 구간을 찾는다.  
- TF‑IDF 기반 텍스트 분류에서, 어떤 모델들이 “성능/무게/속도” 균형이 좋은지 정리한다.  
- Dense 딥러닝 모델이 전통 ML 대비 가지는 장단점을, 실제 수치와 함께 설명할 수 있게 된다.  
- 향후 로컬 LLM이나 온디바이스 NLP를 설계할 때, “최소한 이 정도 베이스라인(LogReg/SVM + TF‑IDF + 제한된 vocab)은 깔고 간다”라는 감각을 갖게 된다.

## 01_환경 설정 

In [15]:
import sys
print(sys.executable)

/usr/bin/python3


In [21]:
import time
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.datasets import reuters


# 01_데이터 로드 및 디코딩
- index -> text
- DTM , TF-idf 학습데이터 준비

In [23]:
num_words_for_load = 10000
(x_train_idx, y_train), (x_test_idx, y_test) = reuters.load_data(
    num_words=num_words_for_load, test_split=0.2)

word_index = reuters.get_word_index(path="reuters_word_index.json")

index_to_word = {index + 3: word for word, index in word_index.items()}
for index, token in enumerate(("<pad>", "<sos>", "<unk>")):
    index_to_word[index] = token

def decode_seq(seqs, index_to_word):
    decoded = []
    for s in seqs:
        decoded.append(" ".join(index_to_word.get(i, "<unk>") for i in s))
    return decoded

x_train_text = decode_seq(x_train_idx, index_to_word)
x_test_text = decode_seq(x_test_idx, index_to_word)

print("Train samples:", len(x_train_text))
print("Test samples:", len(x_test_text))

Train samples: 8982
Test samples: 2246


## 02_실험설정

In [24]:
vocab_sizes = [2000, 5000, 10000, None] 
results = []  

# 전통 ML / 트리 / 앙상블 모델 (Dense는 따로 처리)
def get_ml_models():
    return {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            multi_class="multinomial",
            solver="lbfgs",
            n_jobs=-1,
        ),
        "LinearSVC": LinearSVC(),
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=-1,
        ),
        "NaiveBayes": MultinomialNB(),
        "LightGBM": LGBMClassifier(
            n_estimators=200,
            learning_rate=0.1,
            num_leaves=31,
            random_state=42,
        ),
        "DecisionTree": DecisionTreeClassifier(
            max_depth=None,
            random_state=42,
        ),
    }


## 03_Vocabulary size 실험

In [ ]:
for vocab in vocab_sizes:
    print("\n","="*60)
    print(f"Vocab Size: {vocab}")
    print("="*60)

    # CountVectorizer 설정
    if vocab is None:
        dtmvector = CountVectorizer()  # All words
        vocab_label = "All"
    else:
        dtmvector = CountVectorizer(max_features=vocab)
        vocab_label = str(vocab)

    tfidf_transformer = TfidfTransformer()

    # 학습 TF-IDF 만들기
    t0 = time.time()
    x_train_dtm = dtmvector.fit_transform(x_train_text)
    x_train_tfidf = tfidf_transformer.fit_transform(x_train_dtm)
    tfidf_train_time = time.time() - t0

    # 테스트 TF-IDF 만들기
    t0 = time.time()
    x_test_dtm = dtmvector.transform(x_test_text)
    x_test_tfidf = tfidf_transformer.transform(x_test_dtm)
    tfidf_test_time = time.time() - t0

    print(f"TF-IDF train shape: {x_train_tfidf.shape}")
    print(f"TF-IDF test  shape: {x_test_tfidf.shape}")

    n_features = x_train_tfidf.shape[1]


Vocab Size: 2000
TF-IDF train shape: (8982, 2000)
TF-IDF test  shape: (2246, 2000)

Vocab Size: 5000
TF-IDF train shape: (8982, 5000)
TF-IDF test  shape: (2246, 5000)

Vocab Size: 10000
TF-IDF train shape: (8982, 9670)
TF-IDF test  shape: (2246, 9670)

Vocab Size: None
TF-IDF train shape: (8982, 9670)
TF-IDF test  shape: (2246, 9670)


## 03-1. 전통 ML + 앙상블 모델들 

In [28]:
ml_models = get_ml_models()
for model_name, model in ml_models.items():
    print(f"\n[ML] Vocab={vocab_label}, Model={model_name}")
    # 학습 시간 측정
    t0 = time.time()
    model.fit(x_train_tfidf, y_train)
    train_time = time.time() - t0
    # 추론 시간 측정
    t0 = time.time()
    y_pred = model.predict(x_test_tfidf)
    test_time = time.time() - t0
    # 성능
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    # 모델 저장 후 파일 크기 측정 (rough model size)
    model_path = f"models/{model_name}_vocab_{vocab_label}.joblib"
    joblib.dump(model, model_path)
    model_size_bytes = os.path.getsize(model_path)
    model_size_kb = model_size_bytes / 1024
    results.append({
        "Vocabulary": vocab_label,
        "Model": model_name,
        "Accuracy": acc,
        "F1_macro": f1_macro,
        "TrainTime_s": train_time,
        "TestTime_s": test_time,
        "ModelSize_KB": model_size_kb,
        "NumFeatures": n_features,
        "TFIDF_TrainTime_s": tfidf_train_time,
        "TFIDF_TestTime_s": tfidf_test_time,
    })



[ML] Vocab=All, Model=LogisticRegression


/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



[ML] Vocab=All, Model=LinearSVC

[ML] Vocab=All, Model=RandomForest

[ML] Vocab=All, Model=XGBoost

[ML] Vocab=All, Model=NaiveBayes

[ML] Vocab=All, Model=LightGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.058110 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 163467
[LightGBM] [Info] Number of data points in the train set: 8982, number of used features: 3966
[LightGBM] [Info] Start training from score -5.095645
[LightGBM] [Info] Start training from score -3.034552
[LightGBM] [Info] Start training from score -4.798913
[LightGBM] [Info] Start training from score -1.044967
[LightGBM] [Info] Start training from score -1.527906
[LightGBM] [Info] Start training from score -6.269765
[LightGBM] [Info] Start training from score -5.231777
[LightGBM] [Info] Start training from score -6.330389
[LightGBM] [Info] Start training from score -4

/usr/local/lib/python3.10/dist-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[ML] Vocab=All, Model=DecisionTree


## 03-2. Dense NN (딥러닝) 모델 실험

In [29]:
print(f"\n[DL] Vocab={vocab_label}, Model=DenseNN")

input_dim = n_features
num_classes = np.max(y_train) + 1  # 46
inputs = Input(shape=(input_dim,))
x = Dense(512, activation="relu")(inputs)
x = Dropout(0.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation="softmax")(x)
dense_model = Model(inputs=inputs, outputs=outputs)
dense_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
dense_model.summary()

# Keras는 sparse matrix 안 받으니 dense로 변환 (주의: 메모리 많이 먹을 수 있음)
x_train_dense = x_train_tfidf.toarray()
x_test_dense = x_test_tfidf.toarray()

# 학습 시간
t0 = time.time()
history = dense_model.fit(
    x_train_dense,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)
train_time = time.time() - t0

# 추론 시간
t0 = time.time()
y_proba = dense_model.predict(x_test_dense, verbose=0)
test_time = time.time() - t0
y_pred = np.argmax(y_proba, axis=1)
acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")

# 모델 파라미터 수 & 파일 크기
num_params = dense_model.count_params()
dense_path = f"models/DenseNN_vocab_{vocab_label}.h5"
dense_model.save(dense_path)
model_size_bytes = os.path.getsize(dense_path)
model_size_kb = model_size_bytes / 1024
results.append({
    "Vocabulary": vocab_label,
    "Model": "DenseNN",
    "Accuracy": acc,
    "F1_macro": f1_macro,
    "TrainTime_s": train_time,
    "TestTime_s": test_time,
    "ModelSize_KB": model_size_kb,
    "NumParams": num_params,
    "NumFeatures": n_features,
    "TFIDF_TrainTime_s": tfidf_train_time,
    "TFIDF_TestTime_s": tfidf_test_time,
})



[DL] Vocab=All, Model=DenseNN


W0000 00:00:1773290189.991336   76886 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1773290189.993880   76886 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1773290190.003912   76886 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13358 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Ti, pci bus id: 0000:01:00.0, compute capability: 12.0a


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 9670)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     4,951,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 46)             │         5,934 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,023,150 (19.16 MB)

...

I0000 00:00:1773290198.256524  107416 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1408__.15


225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4846 - loss: 2.3004

I0000 00:00:1773290204.093473  107924 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_10', 8 bytes spill stores, 8 bytes spill loads



225/225 ━━━━━━━━━━━━━━━━━━━━ 12s 32ms/step - accuracy: 0.6139 - loss: 1.6641 - val_accuracy: 0.7524 - val_loss: 1.0887
Epoch 2/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8149 - loss: 0.8009 - val_accuracy: 0.8008 - val_loss: 0.8763
Epoch 3/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8884 - loss: 0.4701 - val_accuracy: 0.8147 - val_loss: 0.8205
Epoch 4/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9328 - loss: 0.2831 - val_accuracy: 0.8191 - val_loss: 0.8268
Epoch 5/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9485 - loss: 0.2049 - val_accuracy: 0.8125 - val_loss: 0.8828
Epoch 6/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9560 - loss: 0.1686 - val_accuracy: 0.8114 - val_loss: 0.9014
Epoch 7/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9602 - loss: 0.1443 - val_accuracy: 0.8102 - val_loss: 0.9250
Epoch 8/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9599 - loss: 0.1347 - val_accuracy: 0.8141 - va

I0000 00:00:1773290214.115529  108991 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_10', 8 bytes spill stores, 8 bytes spill loads



# 결과 정리

In [30]:
# 04. 결과 정리 --------------------------------------------------------

results_df = pd.DataFrame(results)

# 과제에서 요구한 형태의 핵심 표 (Accuracy / F1만)
pivot_main = results_df.pivot_table(
    index=["Vocabulary", "Model"],
    values=["Accuracy", "F1_macro"],
)

print("\n=== 성능 요약 (Accuracy / F1_macro) ===")
print(pivot_main)

# 효율 지표까지 포함한 전체 결과
print("\n=== 전체 결과 (성능 + 시간 + 모델 크기) ===")
print(results_df)

# 필요하면 CSV로 저장
results_df.to_csv("experiment_results_reuters_tfidf_models.csv", index=False)


=== 성능 요약 (Accuracy / F1_macro) ===
                               Accuracy  F1_macro
Vocabulary Model                                 
All        DecisionTree        0.696794  0.460283
           DenseNN             0.804541  0.615634
           LightGBM            0.399822  0.024397
           LinearSVC           0.829920  0.680843
           LogisticRegression  0.795637  0.472114
           NaiveBayes          0.656723  0.096728
           RandomForest        0.757346  0.447596
           XGBoost             0.810775  0.629394

=== 전체 결과 (성능 + 시간 + 모델 크기) ===
  Vocabulary               Model  Accuracy  F1_macro  TrainTime_s  TestTime_s  \
0        All  LogisticRegression  0.795637  0.472114     4.905563    0.004318   
1        All           LinearSVC  0.829920  0.680843     0.654705    0.002881   
2        All        RandomForest  0.757346  0.447596     6.643259    0.079939   
3        All             XGBoost  0.810775  0.629394   695.891925    0.086490   
4        All          Nai